In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

In [1]:
class MF(object):

    def __init__(self, Y_data, K, lam = 0.1, Xinit = None, Winit = None, 
            learning_rate = 0.5, max_iter = 1000, print_every = 100, user_based = 1):

        self.Y_raw_data = Y_data   # Dữ liệu gốc (user, item, rating)
        self.K = K                 # Số latent features

        # Hệ số regularization (tránh overfitting)
        self.lam = lam

        # Tốc độ học (learning rate)
        self.learning_rate = learning_rate

        # Số vòng lặp tối đa
        self.max_iter = max_iter

        # In kết quả sau mỗi print_every vòng lặp
        self.print_every = print_every

        # Chọn chuẩn hóa theo user hay item
        self.user_based = user_based

        # Số lượng user, item và số rating
        # +1 vì index bắt đầu từ 0
        self.n_users = int(np.max(Y_data[:, 0])) + 1 
        self.n_items = int(np.max(Y_data[:, 1])) + 1
        self.n_ratings = Y_data.shape[0]
        
        # Khởi tạo ma trận item (X)
        if Xinit is None:
            self.X = np.random.randn(self.n_items, K)
        else:
            self.X = Xinit 
        
        # Khởi tạo ma trận user (W)
        if Winit is None: 
            self.W = np.random.randn(K, self.n_users)
        else:
            self.W = Winit
            
        # Bản sao dữ liệu để chuẩn hóa
        self.Y_data_n = self.Y_raw_data.copy()


    
    def normalize_Y(self):
        """
        Chuẩn hóa dữ liệu rating:
        - Nếu user_based: trừ đi mean của từng user
        - Nếu item_based: trừ đi mean của từng item
        """

        if self.user_based:
            user_col = 0
            item_col = 1
            n_objects = self.n_users
        else:
            user_col = 1
            item_col = 0 
            n_objects = self.n_items

        users = self.Y_raw_data[:, user_col] 
        self.mu = np.zeros((n_objects,))

        for n in range(n_objects):
            # Lấy các dòng mà user/item = n
            ids = np.where(users == n)[0].astype(np.int32)

            # Lấy các item liên quan
            item_ids = self.Y_data_n[ids, item_col]

            # Lấy rating tương ứng
            ratings = self.Y_data_n[ids, 2]

            # Tính trung bình
            m = np.mean(ratings) 

            # Nếu không có rating (NaN) thì gán = 0
            if np.isnan(m):
                m = 0 

            self.mu[n] = m

            # Chuẩn hóa: rating - mean
            self.Y_data_n[ids, 2] = ratings - self.mu[n]

    def loss(self):
        """
        Hàm mất mát:
        - Sai số giữa dự đoán và thực tế
        - + regularization
        """
        L = 0 

        for i in range(self.n_ratings):
            # user, item, rating
            n, m, rate = int(self.Y_data_n[i, 0]), int(self.Y_data_n[i, 1]), self.Y_data_n[i, 2]

            # Tính lỗi bình phương
            L += 0.5*(rate - self.X[m, :].dot(self.W[:, n]))**2
        
        # Trung bình
        L /= self.n_ratings

        # Thêm regularization
        L += 0.5*self.lam*(np.linalg.norm(self.X, 'fro') + np.linalg.norm(self.W, 'fro'))

        return L 

    def get_items_rated_by_user(self, user_id):
        ids = np.where(self.Y_data_n[:,0] == user_id)[0] 
        item_ids = self.Y_data_n[ids, 1].astype(np.int32)
        ratings = self.Y_data_n[ids, 2]

        return (item_ids, ratings)
        
        
    def get_users_who_rate_item(self, item_id):
        ids = np.where(self.Y_data_n[:,1] == item_id)[0] 
        user_ids = self.Y_data_n[ids, 0].astype(np.int32)
        ratings = self.Y_data_n[ids, 2]

        return (user_ids, ratings)
    
    def updateX(self):
        """
        Cập nhật ma trận item (X)
        """
        for m in range(self.n_items):
            user_ids, ratings = self.get_users_who_rate_item(m)

            Wm = self.W[:, user_ids]

            # Gradient của X[m]
            grad_xm = -(ratings - self.X[m, :].dot(Wm)).dot(Wm.T)/self.n_ratings + \
                                               self.lam*self.X[m, :]

            # Cập nhật theo gradient descent
            self.X[m, :] -= self.learning_rate*grad_xm.reshape((self.K,))
    

    def updateW(self):
        """
        Cập nhật ma trận user (W)
        """
        for n in range(self.n_users):
            item_ids, ratings = self.get_items_rated_by_user(n)

            Xn = self.X[item_ids, :]

            # Gradient của W[:, n]
            grad_wn = -Xn.T.dot(ratings - Xn.dot(self.W[:, n]))/self.n_ratings + \
                        self.lam*self.W[:, n]

            # Cập nhật
            self.W[:, n] -= self.learning_rate*grad_wn.reshape((self.K,))


    def fit(self):
        self.normalize_Y()

        for it in range(self.max_iter):
            self.updateX()
            self.updateW()

            if (it + 1) % self.print_every == 0:
                rmse_train = self.evaluate_RMSE(self.Y_raw_data)
                print('iter =', it + 1, ', loss =', self.loss(), ', RMSE train =', rmse_train)

    def pred(self, u, i):
        """
        Dự đoán rating của user u cho item i
        """
        u = int(u)
        i = int(i)

        # Bias (mean)
        if self.user_based:
            bias = self.mu[u]
        else: 
            bias = self.mu[i]

        pred = self.X[i, :].dot(self.W[:, u]) + bias 

        # Giới hạn trong khoảng [0, 5]
        if pred < 0:
            return 0 
        if pred > 5: 
            return 5 

        return pred 
        

    def pred_for_user(self, user_id):
        """
        Dự đoán tất cả item mà user chưa đánh giá
        """
        ids = np.where(self.Y_data_n[:, 0] == user_id)[0]

        items_rated_by_u = self.Y_data_n[ids, 1].tolist()              
        
        y_pred = self.X.dot(self.W[:, user_id]) + self.mu[user_id]

        predicted_ratings= []

        for i in range(self.n_items):
            if i not in items_rated_by_u:
                predicted_ratings.append((i, y_pred[i]))
        
        return predicted_ratings
    

    # ================= RMSE =================
    def evaluate_RMSE(self, rate_test):
        """
        Tính RMSE trên tập test
        """
        n_tests = rate_test.shape[0]
        SE = 0  # tổng bình phương sai số

        for n in range(n_tests):
            pred = self.pred(rate_test[n, 0], rate_test[n, 1])
            SE += (pred - rate_test[n, 2])**2 

        RMSE = np.sqrt(SE/n_tests)

        return RMSE

In [6]:
# Kiểm tra trên tập dữ liệu 
ratings_base = pd.read_csv('../ml-100k/ua.base', sep='\t', names = ['user_id', 'movie_id', 'rating', 'unix_timestamp'])
ratings_test = pd.read_csv('../ml-100k/ua.test', sep='\t', names = ['user_id', 'movie_id', 'rating', 'unix_timestamp'])

rate_train = ratings_base.values
rate_test = ratings_test.values

# 
rate_train[:, :2] -= 1
rate_test[:, :2] -= 1

rate_train[:5]

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       [        0,         3,         3, 876893119],
       [        0,         4,         3, 889751712]])

In [7]:
# Chuẩn hóa dựa vào users
rs = MF(rate_train, K = 10, lam = .1, print_every = 10, 
    learning_rate = 0.75, max_iter = 100, user_based = 1)
rs.fit()
# Tính toán RMSE trên tập test
rmse_test = rs.evaluate_RMSE(rate_test)
print('RMSE test =', rmse_test)

iter = 10 , loss = 5.625217507418412 , RMSE train = 1.2055998768955751
iter = 20 , loss = 2.6312086358880444 , RMSE train = 1.0394590595519877
iter = 30 , loss = 1.3424793373541122 , RMSE train = 1.0312016173686773
iter = 40 , loss = 0.7552447247060423 , RMSE train = 1.0308807173361125
iter = 50 , loss = 0.4861168490325738 , RMSE train = 1.0308773391777255
iter = 60 , loss = 0.36270608687923545 , RMSE train = 1.0308797354792287
iter = 70 , loss = 0.3061119559268695 , RMSE train = 1.030880434056867
iter = 80 , loss = 0.2801587168404871 , RMSE train = 1.0308806016157464
iter = 90 , loss = 0.2682569456730624 , RMSE train = 1.0308806402986463
iter = 100 , loss = 0.2627989710665036 , RMSE train = 1.030880649118545
RMSE test = 1.043134961768087


In [8]:
# Chuẩn hóa dựa vào items
rs = MF(rate_train, K = 10, lam = .1, print_every = 10, learning_rate = 0.75, max_iter = 100, user_based = 0)
rs.fit()

# Tính toán RMSE dựa trên tập test
rmse_test = rs.evaluate_RMSE(rate_test)
print('RMSE test =', rmse_test)

e:\Downloads\New folder\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\Downloads\New folder\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


iter = 10 , loss = 5.641601583611829 , RMSE train = 1.1757918749492158
iter = 20 , loss = 2.6275776649986846 , RMSE train = 1.0055967017981833
iter = 30 , loss = 1.329700855435878 , RMSE train = 0.9971141543009264
iter = 40 , loss = 0.7382851374416323 , RMSE train = 0.9967928744124239
iter = 50 , loss = 0.46724332947675795 , RMSE train = 0.9967914559330753
iter = 60 , loss = 0.34295572323207296 , RMSE train = 0.9967943409000982
iter = 70 , loss = 0.28595971289662014 , RMSE train = 0.9967951427412793
iter = 80 , loss = 0.2598222467726302 , RMSE train = 0.9967953314304931
iter = 90 , loss = 0.24783601585040066 , RMSE train = 0.9967953744032872
iter = 100 , loss = 0.24233931876050419 , RMSE train = 0.9967953840916488
RMSE test = 1.0426526216649903


In [9]:
rs = MF(rate_train, K = 2, lam = 0, print_every = 10, learning_rate = 1, max_iter = 100, user_based = 0)
rs.fit()

# Tính toán RMSE dựa trên tập test
rmse_test = rs.evaluate_RMSE(rate_test)
print('RMSE test =', rmse_test)

e:\Downloads\New folder\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\Downloads\New folder\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


iter = 10 , loss = 1.3702021568273144 , RMSE train = 1.5411780249704086
iter = 20 , loss = 1.2820851077432274 , RMSE train = 1.5176591405389899
iter = 30 , loss = 1.2047011263850638 , RMSE train = 1.4956679676079365
iter = 40 , loss = 1.1363314236353717 , RMSE train = 1.4749964246943592
iter = 50 , loss = 1.0755915308905195 , RMSE train = 1.4555100944424624
iter = 60 , loss = 1.0213557588418796 , RMSE train = 1.4371699289989823
iter = 70 , loss = 0.9727009203365111 , RMSE train = 1.4199594365861392
iter = 80 , loss = 0.928863884931561 , RMSE train = 1.4037699968251245
iter = 90 , loss = 0.8892092011084871 , RMSE train = 1.3884641553292565
iter = 100 , loss = 0.8532041390077678 , RMSE train = 1.3740118905646643
RMSE test = 1.4460834172241968
